# Convert GTFS frequencies to stop_times

Converts GTFS with frequencies to explicit schedules (compatible with r5r/r5py).

In [ ]:
# Load parameters from params.json
library(jsonlite)

# Read params.json
params <- fromJSON("params.json")

# Extract city parameter
ciudad <- params$city
print(paste("City:", ciudad))

In [ ]:
# Assign memory to the JVM (adjust the size as needed)
options(java.parameters = "-Xmx8G")   # 8 GB; you can use 6G, 4G, etc.

# (optional) check that it was saved in the options
getOption("java.parameters")

# 1) Required packages
install.packages(c("gtfstools", "fs"))   # fs for file handling
library(gtfstools)
library(fs)
library(r5r)

In [ ]:
# 2) Paths
path_gtfs_por_frecuencias  <- path( paste0("../data/", ciudad, "/gtfs-frequencies.zip"))        # original file, must be compressed manually
path_gtfs_convertido <- path( paste0("../data/", ciudad, "/gtfs_stop_times.zip"))        # output

In [ ]:
# 3) Read GTFS
gtfs <- read_gtfs(path_gtfs_por_frecuencias)

# Fix data types in frequencies table if it exists
if ("frequencies" %in% names(gtfs)) {
  # Ensure headway_secs and exact_times are integers
  gtfs$frequencies$headway_secs <- as.integer(gtfs$frequencies$headway_secs)
  gtfs$frequencies$exact_times <- as.integer(gtfs$frequencies$exact_times)
  
  message("Converting frequencies -> stop_times… (may take a while)")
  
  # Expands each headway interval to trips with exact schedules
  # Uses relative times from stop_times of the base trip.
  gtfs <- frequencies_to_stop_times(gtfs)
  
  # Remove the frequencies table from the object to prevent r5r from failing
  gtfs$frequencies <- NULL
} else {
  message("This GTFS does not have frequencies; nothing to convert.")
}

# 5) (Optional) Validate order
# Ensure stop_times is sorted by trip_id, stop_sequence:
gtfs$stop_times <- gtfs$stop_times[
  order(gtfs$stop_times$trip_id, gtfs$stop_times$stop_sequence),
]

# 6) Save converted GTFS
write_gtfs(gtfs, path_gtfs_convertido)
message("Expanded GTFS saved at: ", path_gtfs_convertido)